# Pythia Family — BASP vs WANDA structured MLP pruning

This notebook applies the **critiPrune** framework to the [Pythia](https://github.com/EleutherAI/pythia) transformer family (14M–6.9B), testing whether structured MLP-neuron pruning shows the universal sigmoid phase transition, **and comparing two importance criteria** for choosing which intermediate neurons to keep:

- **WANDA** ([Sun et al., 2023](https://arxiv.org/abs/2306.11695)) — the *input-side* score $\;\mathrm{in}_j=\sum_k |W^{\text{up}}_{jk}|\,\lVert x_k\rVert\;$: how strongly neuron $j$ is **driven** by the calibration activations.
- **BASP** (this project, see `.docs/lessons_about-pruning.md`) — the **bidirectional** product $\;s_j=\mathrm{in}_j\cdot\mathrm{out}_j\;$ with output relevance $\;\mathrm{out}_j=\lVert W^{\text{down}}_{:,j}\rVert_2\,\lVert a_j\rVert\;$: a neuron matters only if it is both **driven** by the input *and* actually **reaches** the residual stream through the down-projection. BASP adds the output-side factor WANDA omits — the structured analogue of the unstructured $|W|\cdot\lVert x\rVert\cdot\lVert a\rVert$ score.

Both keep the top-$s$ fraction of $d_{\text{ff}}$ neurons **per layer** (uniform-$s$ structured allocation), one-shot, no fine-tuning. The figure of merit is the **critical density $s_0$** — the lower, the more prunable; BASP is expected to push $s_0$ below WANDA, with the margin growing with depth.

> **Notation.** Throughout we use $s$ for the retention rate (fraction of neurons kept), consistent with the rest of the repo. (Earlier drafts called this $K$.)

---

## Pipeline (per model)

1. **Calibration** — collect MLP activation statistics over 128 C4 samples: input norms $\lVert x_k\rVert$ and post-GELU intermediate norms $\lVert a_j\rVert$.
2. **Scoring** — score every intermediate neuron with **both** WANDA and BASP.
3. **$s$-sweep** — for each density $s\in(0,1]$, keep the top-$s$ neurons per layer (a forward hook zeroes the rest after GELU) and measure WikiText-2 perplexity.
4. **Sigmoid fit** — fit the recovery curve $A(s)=A_0+(A_\infty-A_0)/(1+e^{-\beta(s-s_0)})$ and extract the critical density $s_0$ and steepness $\beta$, for each method.
5. **Scaling laws** — fit $s_0=c\cdot d_{\text{ff}}^{\alpha}\cdot L^{\gamma}$ across the family, per method.

## Requirements

- **GPU**: Colab T4 handles up to ~1.4B; A100 for 2.8B/6.9B. *Comparing two methods doubles the sweep cost* — trim `--models` if needed.
- **Dependencies**: `torch`, `transformers`, `datasets`, `accelerate`, `scipy`, `numpy`, `matplotlib`
- **HuggingFace token**: prompted once via `getpass` (or set the `HF_TOKEN` env var).

## Init

In [ ]:
"""
BASP vs WANDA — structured MLP pruning across the Pythia model family
====================================================================
Tests whether the sigmoid pruning law and scaling relations discovered on FC
networks hold for transformer LLMs, and compares two importance criteria for
choosing which MLP intermediate neurons to keep:

  * WANDA (Sun et al. 2023): input-side score in_j = sum_k |W_up[j,k]| * ||x_k||
    -- how strongly each neuron is *driven* by the calibration activations.
  * BASP  (this project, .docs/lessons_about-pruning.md): the bidirectional
    product s_j = in_j * out_j with output relevance
    out_j = ||W_down[:,j]||_2 * ||a_j|| -- how much the neuron's activation
    actually reaches the residual stream. BASP adds the output-side factor
    WANDA omits (the structured analogue of |W|*||x||*||a||).

Both keep the top-s fraction of d_ff neurons *per layer* (uniform-s structured
allocation), applied one-shot with no fine-tuning. Lower critical density s_0
= more prunable = better.

Run on Google Colab with a GPU runtime (T4 sufficient for 14M-1.4B).

Usage
-----
  !pip install transformers datasets accelerate scipy matplotlib -q
  # then run all cells; main() prompts once for a HuggingFace token
  # (or set os.environ['HF_TOKEN']). Select models via --models or DEFAULT_MODELS.

Method (per model)
------------------
  1. Collect MLP activation stats over N calib samples (C4): ||x_k||, ||a_j||.
  2. Score every intermediate neuron with WANDA and with BASP.
  3. For each density s, keep the top-s neurons per layer (forward hook zeroes
     the rest after GELU) and measure WikiText-2 perplexity.
  4. Fit recovery(s) = A0 + (Ainf-A0)/(1+exp(-beta*(s-s0))); extract s0, beta.
  5. Fit joint scaling laws s0 = c * d_ff^alpha * L^gamma, per method.

Output
------
  pythia_figures/
    pythia_recovery_curves.png   -- WANDA vs BASP sigmoid fits per model
    pythia_scaling_laws.png      -- s0 / beta / g vs architecture, per method
    pythia_parameter_table.png   -- summary comparison table
    pythia_results.json          -- all numerical results
"""

import os, sys, json, time, gc, argparse, warnings
from getpass import getpass
warnings.filterwarnings('ignore')

import numpy as np
from scipy.optimize import curve_fit

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ========================
#  CONFIGURATION
# ========================

OUTPUT_DIR = './pythia_figures'

# Pythia model family: (name_suffix, L, d_model, d_ff)
# d_ff = 4 * d_model for all Pythia models
PYTHIA_MODELS = {
    '14m':  dict(hf='EleutherAI/pythia-14m',   L=6,  H=128,   d_ff=512),
    '31m':  dict(hf='EleutherAI/pythia-31m',   L=6,  H=256,   d_ff=1024),
    '70m':  dict(hf='EleutherAI/pythia-70m',   L=6,  H=512,   d_ff=2048),
    '160m': dict(hf='EleutherAI/pythia-160m',  L=12, H=768,   d_ff=3072),
    '410m': dict(hf='EleutherAI/pythia-410m',  L=24, H=1024,  d_ff=4096),
    '1b':   dict(hf='EleutherAI/pythia-1b',    L=16, H=2048,  d_ff=8192),
    '1.4b': dict(hf='EleutherAI/pythia-1.4b',  L=24, H=2048,  d_ff=8192),
    '2.8b': dict(hf='EleutherAI/pythia-2.8b',  L=32, H=2560,  d_ff=10240),
    '6.9b': dict(hf='EleutherAI/pythia-6.9b',  L=32, H=4096,  d_ff=16384),
}

# Default models (fit on T4 16GB comfortably). Two methods double the cost.
DEFAULT_MODELS = ['14m', '31m', '70m', '160m', '410m', '1b', '1.4b']

# Importance criteria to compare (see the scoring cell)
METHODS = ['wanda', 'basp']

# Density sweep: fraction s of d_ff neurons KEPT per layer (the retention rate).
# Dense at both ends, fine resolution at low s where the transition lives.
S_VALUES = np.unique(np.sort(np.concatenate([
    np.arange(0.01, 0.06, 0.01),        # 1%-5%   (fine resolution at low s)
    np.arange(0.05, 0.20, 0.025),       # 5%-20%
    np.arange(0.20, 0.55, 0.05),        # 20%-50%
    np.arange(0.50, 1.01, 0.10),        # 50%-100%
])))

# Calibration
N_CALIB_SAMPLES = 128
CALIB_SEQ_LEN   = 128

# Perplexity evaluation
EVAL_DATASET    = 'wikitext'          # 'wikitext' or 'pile-10k'
EVAL_MAX_TOKENS = 40_000             # cap for speed; increase for precision
EVAL_STRIDE     = 1024
EVAL_MAX_LENGTH = 2048


## Pruning

### Activation Collector

In [ ]:
class MLPActivationCollector:
    """Collect squared activation norms at MLP up-projection inputs
    and post-GELU intermediate activations."""

    def __init__(self, model):
        self.model = model
        self.n_layers = model.config.num_hidden_layers
        self.hooks = []
        self.input_sq = {}     # layer_idx -> [d_model]
        self.inter_sq = {}     # layer_idx -> [d_ff]
        self.n_tokens = 0

    def _hook_input(self, idx):
        def fn(module, inp, out):
            x = inp[0].detach().float().reshape(-1, inp[0].shape[-1])
            if idx not in self.input_sq:
                self.input_sq[idx] = torch.zeros(x.shape[1], device=x.device)
            self.input_sq[idx] += (x ** 2).sum(dim=0)
            if idx == 0:
                self.n_tokens += x.shape[0]
        return fn

    def _hook_inter(self, idx):
        def fn(module, inp, out):
            # inp to dense_4h_to_h = post-GELU activations
            x = inp[0].detach().float().reshape(-1, inp[0].shape[-1])
            if idx not in self.inter_sq:
                self.inter_sq[idx] = torch.zeros(x.shape[1], device=x.device)
            self.inter_sq[idx] += (x ** 2).sum(dim=0)
        return fn

    def register(self):
        for i, layer in enumerate(self.model.gpt_neox.layers):
            self.hooks.append(
                layer.mlp.dense_h_to_4h.register_forward_hook(self._hook_input(i)))
            self.hooks.append(
                layer.mlp.dense_4h_to_h.register_forward_hook(self._hook_inter(i)))

    def remove(self):
        for h in self.hooks:
            h.remove()
        self.hooks.clear()

    def get_norms(self):
        """Return RMS norms (not raw squared sums)."""
        input_norms, inter_norms = {}, {}
        for i in range(self.n_layers):
            input_norms[i] = torch.sqrt(self.input_sq[i] / self.n_tokens)
            inter_norms[i] = torch.sqrt(self.inter_sq[i] / self.n_tokens)
        return input_norms, inter_norms

### Scoring — WANDA (input-side) vs BASP (bidirectional)

In [ ]:
def compute_wanda_scores(model, input_norms, inter_norms):
    """Standard WANDA (Sun et al. 2023), structured to MLP neurons.

    Each intermediate neuron j is produced by row j of the up-projection. Its
    WANDA importance is the input-activation-weighted L1 norm of that row,

        in_j = sum_k |W_up[j, k]| * ||x_k||          (the *input drive*),

    i.e. the input side only -- ||a_j|| and the down-projection are NOT used.
    This is the faithful structured WANDA baseline. Returns {layer: Tensor[d_ff]}.
    """
    scores = {}
    for i, layer in enumerate(model.gpt_neox.layers):
        W_up = layer.mlp.dense_h_to_4h.weight.data.float()       # [d_ff, d_model]
        scores[i] = (W_up.abs() * input_norms[i].unsqueeze(0)).sum(dim=1)  # [d_ff]
    return scores


def compute_basp_scores(model, input_norms, inter_norms):
    """BASP bidirectional neuron score (this project, .docs/lessons_about-pruning.md).

    Each intermediate neuron j sits on a path
        residual --(W_up)--> GELU --(W_down)--> residual,
    and BASP weights it by *both* ends of that path:

        input drive       in_j  = sum_k |W_up[j, k]| * ||x_k||      (= WANDA)
        output relevance  out_j = ||W_down[:, j]||_2 * ||a_j||      (down-proj x act)
        BASP score         s_j  = in_j * out_j

    A neuron survives only if it is both strongly *driven* by the input AND
    actually *reaches* the output through the down-projection -- the structured
    analogue of the unstructured |W|*||x||*||a|| score. One-shot, label-free.
    Returns {layer: Tensor[d_ff]}.
    """
    scores = {}
    for i, layer in enumerate(model.gpt_neox.layers):
        W_up = layer.mlp.dense_h_to_4h.weight.data.float()       # [d_ff, d_model]
        W_down = layer.mlp.dense_4h_to_h.weight.data.float()     # [d_model, d_ff]
        in_drive = (W_up.abs() * input_norms[i].unsqueeze(0)).sum(dim=1)   # [d_ff]
        out_relevance = W_down.norm(p=2, dim=0) * inter_norms[i]           # [d_ff]
        scores[i] = in_drive * out_relevance
    return scores


# Dispatch table: method name -> scoring function (same signature)
SCORERS = {
    'wanda': compute_wanda_scores,
    'basp':  compute_basp_scores,
}


### Top-K via Hooks

In [ ]:
class TopKPruner:
    """Structured MLP-neuron pruning from a precomputed per-layer score dict.

    Keeps the top-``s`` fraction of d_ff neurons in each layer (uniform-s
    per-layer allocation, identical for WANDA and BASP -- only the *score*
    differs). The non-kept neurons are zeroed right after GELU by a forward
    hook, removing their contribution to the down-projection output.

    Works with any score dict {layer_idx: Tensor[d_ff]}.
    """

    def __init__(self, model, scores, density=1.0):
        self.model = model
        self.scores = scores
        self.d_ff = model.config.intermediate_size
        self.density = density
        self._handles = []

        k = max(1, int(round(self.d_ff * density)))   # neurons kept per layer
        self.masks = {}
        for i in range(model.config.num_hidden_layers):
            s = scores[i]
            _, top_idx = torch.topk(s, k)
            mask = torch.zeros(self.d_ff, device=s.device, dtype=torch.float16)
            mask[top_idx] = 1.0
            self.masks[i] = mask  # [d_ff]

    def _make_hook(self, layer_idx):
        mask = self.masks[layer_idx]
        def hook_fn(module, inp, out):
            return out * mask.unsqueeze(0).unsqueeze(0)
        return hook_fn

    def enable(self):
        for i, layer in enumerate(self.model.gpt_neox.layers):
            h = layer.mlp.act.register_forward_hook(self._make_hook(i))
            self._handles.append(h)

    def disable(self):
        for h in self._handles:
            h.remove()
        self._handles.clear()

    def __enter__(self):
        self.enable()
        return self

    def __exit__(self, *args):
        self.disable()


### Perplexity Evaluation

In [ ]:
def evaluate_perplexity(model, tokenizer, dataset='wikitext',
                        max_tokens=40_000, stride=1024,
                        max_length=2048, device='cuda'):
    """Sliding-window perplexity on WikiText-2 or Pile-10k subset."""
    from datasets import load_dataset

    if dataset == 'wikitext':
        # Canonical repo id (recent `datasets` rejects the bare 'wikitext').
        test = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1',
                            split='test')
        text = '\n\n'.join(test['text'])
    else:
        ds = load_dataset('NeelNanda/pile-10k', split='train')
        text = '\n\n'.join(ds['text'][:300])

    encodings = tokenizer(text, return_tensors='pt')
    seq_len = min(encodings.input_ids.size(1), max_tokens)

    nll_sum = 0.0
    n_tokens = 0
    prev_end = 0

    for begin in range(0, seq_len, stride):
        end = min(begin + max_length, seq_len)
        trg_len = end - prev_end
        input_ids = encodings.input_ids[:, begin:end].to(device)
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100

        with torch.no_grad():
            loss = model(input_ids, labels=target_ids).loss

        num_scored = (target_ids != -100).sum().item() - 1
        if num_scored > 0:
            nll_sum += loss.float().item() * num_scored
            n_tokens += num_scored

        prev_end = end
        if end >= seq_len:
            break

    if n_tokens == 0:
        return float('inf')
    ppl = np.exp(nll_sum / n_tokens)
    return ppl

### Sigmoid Fit

In [ ]:
def sigmoid_fn(s, A_inf, A_0, s_0, beta):
    s = np.asarray(s, dtype=float)
    return A_0 + (A_inf - A_0) / (1.0 + np.exp(
        np.clip(-beta * (s - s_0), -500, 500)))


def fit_sigmoid(s_values, recoveries):
    """
    Fit recovery(s) = A0 + (Ainf - A0)/(1 + exp(-beta*(s - s0)))
    where s is the fraction of neurons kept (0 to 1).

    Returns (popt, R2) or (None, None).  popt = (A_inf, A_0, s_0, beta).
    """
    s_arr = np.array(s_values)
    r_arr = np.array(recoveries)

    try:
        p0 = [max(r_arr), min(r_arr), np.median(s_arr), 10.0]
        bounds = (
            [0.0, -0.1, 0.0, 0.1],
            [1.5, 1.0, 1.0, 200.0],
        )
        popt, pcov = curve_fit(sigmoid_fn, s_arr, r_arr, p0=p0,
                                bounds=bounds, maxfev=30000)
        resid = r_arr - sigmoid_fn(s_arr, *popt)
        ss_res = np.sum(resid ** 2)
        ss_tot = np.sum((r_arr - r_arr.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
        return popt, r2
    except Exception as e:
        print(f"    Sigmoid fit failed: {e}")
        return None, None


## Experiments

### Single Model Experiment

In [ ]:
def run_single_model(model_key, token, device='cuda',
                     eval_max_tokens=EVAL_MAX_TOKENS,
                     n_calib_samples=N_CALIB_SAMPLES):
    """
    Full pipeline for one Pythia model, comparing every method in METHODS:
    load -> calibrate -> score (all methods) -> sweep s (per method) ->
    fit sigmoid (per method) -> clean up.
    """
    spec = PYTHIA_MODELS[model_key]
    hf_name = spec['hf']
    d_ff = spec['d_ff']

    print(f"\n{'═'*60}")
    print(f"  {hf_name}  (L={spec['L']}, H={spec['H']}, d_ff={d_ff})")
    print(f"{'═'*60}")

    # --- Load model --------------------------------------
    t0 = time.time()
    print(f"  Loading model ...", end='', flush=True)
    model = AutoModelForCausalLM.from_pretrained(
        hf_name, torch_dtype=torch.float16, token=token,
        low_cpu_mem_usage=True).to(device)
    model.eval()
    tokenizer = AutoTokenizer.from_pretrained(hf_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    n_params_M = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"  [{time.time()-t0:.0f}s]")

    # --- Baseline perplexity --------------------------------
    print(f"  Baseline perplexity ...", end='', flush=True)
    t1 = time.time()
    ppl_baseline = evaluate_perplexity(
        model, tokenizer, dataset=EVAL_DATASET,
        max_tokens=eval_max_tokens, stride=EVAL_STRIDE,
        max_length=EVAL_MAX_LENGTH, device=device)
    print(f"  {ppl_baseline:.2f}  [{time.time()-t1:.0f}s]")

    # -- Calibration: collect activation norms -------------------
    print(f"  Collecting activations ({n_calib_samples} samples) ...", end='', flush=True)
    t1 = time.time()
    from datasets import load_dataset
    cal_texts = []
    try:
        c4 = load_dataset('allenai/c4', 'en', split='train', streaming=True)
        for sample in c4:
            cal_texts.append(sample['text'])
            if len(cal_texts) >= n_calib_samples:
                break
    except Exception as e:
        print(f"  (C4 unavailable: {e}; calibrating on wikitext-2 train)",
              end='', flush=True)
        wt = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='train')
        for t in wt['text']:
            if t.strip():
                cal_texts.append(t)
            if len(cal_texts) >= n_calib_samples:
                break

    collector = MLPActivationCollector(model)
    collector.register()
    with torch.no_grad():
        for text in cal_texts:
            ids = tokenizer(text, return_tensors='pt', truncation=True,
                            max_length=CALIB_SEQ_LEN).to(device)
            model(**ids)
    collector.remove()
    input_norms, inter_norms = collector.get_norms()
    del collector
    print(f"  [{time.time()-t1:.0f}s]")

    # -- Neuron scores for every method ----------------------------
    print(f"  Computing scores: {', '.join(METHODS)} ...", end='', flush=True)
    t1 = time.time()
    method_scores = {m: SCORERS[m](model, input_norms, inter_norms)
                     for m in METHODS}
    del input_norms, inter_norms
    print(f"  [{time.time()-t1:.0f}s]")

    result = {
        'model': model_key,
        'hf_name': hf_name,
        'L': spec['L'],
        'H': spec['H'],
        'd_ff': d_ff,
        'n_params_M': n_params_M,
        'ppl_baseline': float(ppl_baseline),
        's_values': [float(s) for s in S_VALUES],
        'methods': {},
    }
    loss_baseline = np.log(ppl_baseline)

    # -- Sweep s for each method ------------------------------------
    for method in METHODS:
        scores = method_scores[method]
        print(f"\n  [{method.upper()}] sweeping {len(S_VALUES)} densities ...")
        ppls, recoveries = [], []
        for s in S_VALUES:
            t1 = time.time()
            pruner = TopKPruner(model, scores, density=float(s))
            pruner.enable()
            ppl = evaluate_perplexity(
                model, tokenizer, dataset=EVAL_DATASET,
                max_tokens=eval_max_tokens, stride=EVAL_STRIDE,
                max_length=EVAL_MAX_LENGTH, device=device)
            pruner.disable()

            # Recovery = baseline_loss / sparse_loss in (0, 1]; 1 at dense.
            loss_sparse = np.log(max(ppl, 1.01))
            recovery = min(loss_baseline / loss_sparse, 1.0)

            ppls.append(float(ppl))
            recoveries.append(float(recovery))
            n_keep = int(round(d_ff * s))
            print(f"    s={s:5.1%} ({n_keep:>5}/{d_ff})  "
                  f"ppl={ppl:>10.2f}  recovery={recovery:.4f}  "
                  f"[{time.time()-t1:.0f}s]")

        popt, r2 = fit_sigmoid(result['s_values'], recoveries)
        mres = {'ppls': ppls, 'recoveries': recoveries}
        if popt is not None:
            A_inf, A_0, s_0, beta = popt
            mres.update({
                'sigmoid_A_inf': float(A_inf),
                'sigmoid_A_0': float(A_0),
                'sigmoid_s_0': float(s_0),
                'sigmoid_s0_neurons': int(round(s_0 * d_ff)),
                'sigmoid_beta': float(beta),
                'sigmoid_g_eff': float(np.exp(-beta)),
                'sigmoid_R2': float(r2),
            })
            print(f"  [{method.upper()}] s0={s_0:.3f} "
                  f"({int(round(s_0*d_ff))}/{d_ff})  beta={beta:.2f}  "
                  f"g={np.exp(-beta):.4f}  R2={r2:.4f}")
        else:
            mres['sigmoid_R2'] = None
            print(f"  [{method.upper()}] sigmoid fit FAILED")
        result['methods'][method] = mres

    # -- Head-to-head s0 for this model -----------------------------
    s0s = {m: result['methods'][m].get('sigmoid_s_0')
           for m in METHODS if result['methods'][m].get('sigmoid_s_0') is not None}
    if len(s0s) == len(METHODS):
        best = min(s0s, key=s0s.get)
        print(f"\n  -> lowest s0: {best.upper()} "
              f"({', '.join(f'{m}={v:.3f}' for m, v in s0s.items())})")

    # -- Cleanup ------------
    del model, tokenizer, method_scores
    gc.collect()
    torch.cuda.empty_cache()

    return result


### Scaling Law fits

In [ ]:
def power_law_2d(HL, a, alpha, gamma):
    H, L = HL
    return a * np.power(H, alpha) * np.power(L, gamma)


def fit_scaling_laws(results, method):
    """Fit s0 / beta / g_eff as power laws in (d_ff, L) for one method."""
    good = [r for r in results
            if r['methods'].get(method, {}).get('sigmoid_R2') is not None
            and r['methods'][method]['sigmoid_R2'] > 0.80]
    if len(good) < 3:
        print(f"  [{method}] too few good fits ({len(good)}) for scaling law")
        return None

    H = np.array([r['d_ff'] for r in good], dtype=float)   # d_ff as "width"
    L = np.array([r['L'] for r in good], dtype=float)
    s0 = np.array([r['methods'][method]['sigmoid_s_0'] for r in good])
    beta = np.array([r['methods'][method]['sigmoid_beta'] for r in good])
    g = np.array([r['methods'][method]['sigmoid_g_eff'] for r in good])

    scaling = {}
    print(f"\n{'═'*60}")
    print(f"  SCALING LAW ANALYSIS — {method.upper()} ({len(good)} models)")
    print(f"{'═'*60}")

    for name, arr, p0, bnd in [
        ('s_0', s0, [0.3, -0.3, 0.3], ([0, -3, -3], [10, 3, 3])),
        ('beta', beta, [100, -0.5, -0.5], ([0, -3, -3], [1e6, 3, 3])),
        ('g_eff', g, [0.5, 0.1, 0.1], ([0, -3, -3], [2, 3, 3])),
    ]:
        try:
            popt, pcov = curve_fit(power_law_2d, (H, L), arr,
                                    p0=p0, bounds=bnd, maxfev=10000)
            pred = power_law_2d((H, L), *popt)
            ss_res = np.sum((arr - pred) ** 2)
            ss_tot = np.sum((arr - arr.mean()) ** 2)
            r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
            perr = np.sqrt(np.diag(pcov))
            a, al, ga = popt
            sym = {'s_0': 's0', 'beta': 'beta', 'g_eff': 'g_eff'}[name]
            scaling[name] = dict(a=float(a), alpha=float(al), gamma=float(ga),
                                  R2=float(r2))
            print(f"\n  {sym} = {a:.4f} × d_ff^{al:.3f} × L^{ga:.3f}")
            print(f"       ± ({perr[0]:.3f}, {perr[1]:.3f}, {perr[2]:.3f})")
            print(f"       R² = {r2:.4f}")
        except Exception as e:
            print(f"  {name} fit failed: {e}")

    print(f"\n  s0 (fractional) statistics:")
    print(f"    mean = {s0.mean():.3f} ± {s0.std():.3f}")
    print(f"    range = [{s0.min():.3f}, {s0.max():.3f}]")
    print(f"{'═'*60}")

    return scaling


### Visualization

In [ ]:
METHOD_STYLE = {
    'wanda': dict(ls='-',  marker='o', label='WANDA'),
    'basp':  dict(ls='--', marker='s', label='BASP'),
}


def _good(results, method):
    return [r for r in results
            if r['methods'].get(method, {}).get('sigmoid_R2') is not None
            and r['methods'][method]['sigmoid_R2'] > 0.80]


def make_plots(results, scalings, output_dir):
    if len(results) < 1:
        print("  No results to plot"); return []
    os.makedirs(output_dir, exist_ok=True)

    models_sorted = sorted(results, key=lambda x: x['n_params_M'])
    colors = plt.cm.viridis(np.linspace(0.1, 0.9, max(len(models_sorted), 1)))
    cmap = {r['model']: colors[i] for i, r in enumerate(models_sorted)}
    paths = []

    # ---------------------------------------------------------------
    #  Figure 1: recovery overlay (WANDA solid, BASP dashed) + s0 bars
    # ---------------------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    ax = axes[0]
    for r in models_sorted:
        s_vals = np.array(r['s_values'])
        for method in METHODS:
            md = r['methods'].get(method, {})
            if 'recoveries' not in md:
                continue
            st = METHOD_STYLE[method]
            ax.scatter(s_vals * 100, md['recoveries'], s=16,
                       color=cmap[r['model']], alpha=0.5,
                       marker=st['marker'], zorder=4)
            if md.get('sigmoid_R2') and md['sigmoid_R2'] > 0.80:
                sf = np.linspace(0.01, 1.0, 300)
                fit = sigmoid_fn(sf, md['sigmoid_A_inf'], md['sigmoid_A_0'],
                                 md['sigmoid_s_0'], md['sigmoid_beta'])
                ax.plot(sf * 100, fit, color=cmap[r['model']], lw=1.8,
                        ls=st['ls'], alpha=0.9)
    model_handles = [plt.Line2D([0], [0], color=cmap[r['model']], lw=3,
                                label=r['model']) for r in models_sorted]
    method_handles = [plt.Line2D([0], [0], color='gray', lw=2,
                                 ls=METHOD_STYLE[m]['ls'],
                                 marker=METHOD_STYLE[m]['marker'],
                                 label=METHOD_STYLE[m]['label']) for m in METHODS]
    leg1 = ax.legend(handles=model_handles, fontsize=8, loc='lower right',
                     title='model')
    ax.add_artist(leg1)
    ax.legend(handles=method_handles, fontsize=9, loc='upper left',
              title='method')
    ax.set_xlabel('MLP neurons kept  $s$ (%)', fontsize=12)
    ax.set_ylabel('Loss recovery (baseline_loss / sparse_loss)', fontsize=11)
    ax.set_title('Pruning recovery: WANDA (—) vs BASP (– –)', fontsize=12)
    ax.grid(alpha=0.3); ax.set_xlim(0, 105)

    ax = axes[1]
    labels = [r['model'] for r in models_sorted]
    x = np.arange(len(labels)); w = 0.38
    for mi, method in enumerate(METHODS):
        vals = [(r['methods'].get(method, {}).get('sigmoid_s_0') or np.nan) * 100
                for r in models_sorted]
        ax.bar(x + (mi - 0.5) * w, vals, w, label=METHOD_STYLE[method]['label'],
               edgecolor='black', lw=0.5, alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('$s_0$ (% of MLP neurons)', fontsize=11)
    ax.set_title('Critical density $s_0$ — lower is better', fontsize=12)
    ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    p = os.path.join(output_dir, 'pythia_recovery_curves.png')
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.close()
    paths.append(p); print(f"  Saved: {p}")

    # ---------------------------------------------------------------
    #  Figure 2: scaling of s0 / beta / g_eff vs d_ff, both methods
    # ---------------------------------------------------------------
    fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
    fig.suptitle('Scaling across the Pythia family: WANDA vs BASP',
                 fontsize=13, y=1.03)
    for col, (key, ylabel, title, logy) in enumerate([
        ('sigmoid_s_0',  '$s_0$ (fraction)',         'Critical density vs $d_{ff}$', True),
        ('sigmoid_beta', r'$\beta$',                 'Steepness vs $d_{ff}$',        True),
        ('sigmoid_g_eff','$g_{eff}=e^{-\\beta}$',    'Coupling vs $d_{ff}$',         False),
    ]):
        ax = axes[col]
        for method in METHODS:
            g = _good(results, method)
            if not g:
                continue
            dff = np.array([r['d_ff'] for r in g], float)
            yv = np.array([r['methods'][method][key] for r in g], float)
            order = np.argsort(dff)
            st = METHOD_STYLE[method]
            ax.scatter(dff, yv, s=60, marker=st['marker'],
                       edgecolors='black', lw=0.5, zorder=5,
                       label=st['label'])
            ax.plot(dff[order], yv[order], ls=st['ls'], lw=1.2, alpha=0.6)
        ax.set_xlabel('$d_{ff}$ (MLP intermediate dim)', fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.set_title(title); ax.set_xscale('log')
        if logy:
            ax.set_yscale('log')
        ax.grid(alpha=0.3, which='both'); ax.legend(fontsize=9)

    plt.tight_layout()
    p = os.path.join(output_dir, 'pythia_scaling_laws.png')
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.close()
    paths.append(p); print(f"  Saved: {p}")

    # ---------------------------------------------------------------
    #  Figure 3: head-to-head comparison table
    # ---------------------------------------------------------------
    rows_models = [r for r in models_sorted
                   if all(r['methods'].get(m, {}).get('sigmoid_R2') is not None
                          for m in ('wanda', 'basp'))]
    if rows_models:
        fig, ax = plt.subplots(figsize=(13, max(3, 0.6 * len(rows_models) + 1.5)))
        ax.axis('off')
        ax.set_title('Pythia family — $s_0$ / $\\beta$ / $R^2$: WANDA vs BASP',
                     fontsize=13, pad=20)
        col_labels = ['Model', 'Params', 'L', 'd_ff', 'PPL',
                      's0 WANDA', 's0 BASP', 'Δs0 (W−B)',
                      'β WANDA', 'β BASP', 'R² W', 'R² B']
        table_data = []
        for r in rows_models:
            wd, bd = r['methods']['wanda'], r['methods']['basp']
            ds0 = wd['sigmoid_s_0'] - bd['sigmoid_s_0']
            table_data.append([
                r['model'], f"{r['n_params_M']:.0f}M", str(r['L']), str(r['d_ff']),
                f"{r['ppl_baseline']:.1f}",
                f"{wd['sigmoid_s_0']:.3f}", f"{bd['sigmoid_s_0']:.3f}", f"{ds0:+.3f}",
                f"{wd['sigmoid_beta']:.1f}", f"{bd['sigmoid_beta']:.1f}",
                f"{wd['sigmoid_R2']:.3f}", f"{bd['sigmoid_R2']:.3f}",
            ])
        table = ax.table(cellText=table_data, colLabels=col_labels,
                         loc='center', cellLoc='center')
        table.auto_set_font_size(False); table.set_fontsize(8); table.scale(1.0, 1.6)
        for j in range(len(col_labels)):
            table[0, j].set_facecolor('#4472C4')
            table[0, j].set_text_props(color='white', fontweight='bold')
        for i in range(1, len(table_data) + 1):
            color = '#D9E2F3' if i % 2 == 0 else 'white'
            for j in range(len(col_labels)):
                table[i, j].set_facecolor(color)
        plt.tight_layout()
        p = os.path.join(output_dir, 'pythia_parameter_table.png')
        plt.savefig(p, dpi=150, bbox_inches='tight'); plt.close()
        paths.append(p); print(f"  Saved: {p}")

    return paths


### Main

In [ ]:
def main():
    parser = argparse.ArgumentParser(
        description='Pythia family pruning: BASP vs WANDA')
    parser.add_argument('--models', nargs='*', default=None,
                        help='Model keys (e.g., 70m 410m 1b) or "all"')
    parser.add_argument('--output', default=OUTPUT_DIR)
    parser.add_argument('--eval-tokens', type=int, default=EVAL_MAX_TOKENS)
    parser.add_argument('--n-calib', type=int, default=N_CALIB_SAMPLES)
    args, unknown = parser.parse_known_args()

    eval_max_tokens = args.eval_tokens
    n_calib_samples = args.n_calib

    # Select models
    if args.models is None:
        model_keys = DEFAULT_MODELS
    elif args.models == ['all']:
        model_keys = list(PYTHIA_MODELS.keys())
    else:
        model_keys = [m for m in args.models if m in PYTHIA_MODELS]

    os.makedirs(args.output, exist_ok=True)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # HuggingFace token: env var first, otherwise prompt once (hidden input)
    token = os.environ.get('HF_TOKEN') or getpass('Enter HuggingFace token: ')

    print("=" * 60)
    print("  PYTHIA FAMILY — BASP vs WANDA STRUCTURED MLP PRUNING")
    print("=" * 60)
    print(f"  Device: {device}")
    if device == 'cuda':
        print(f"  GPU: {torch.cuda.get_device_name()}")
    print(f"  Methods: {METHODS}")
    print(f"  Models: {model_keys}")
    print(f"  Densities s: {len(S_VALUES)} levels from "
          f"{S_VALUES[0]:.0%} to {S_VALUES[-1]:.0%}")
    print(f"  Eval tokens: {eval_max_tokens:,}   Calib samples: {n_calib_samples}")
    print(f"  (each model runs one s-sweep per method -> ~{len(METHODS)}x cost)")

    t_total = time.time()
    results = []

    for mk in model_keys:
        try:
            r = run_single_model(mk, token, device=device,
                                 eval_max_tokens=eval_max_tokens,
                                 n_calib_samples=n_calib_samples)
            results.append(r)
        except Exception as e:
            print(f"\n  ✗ {mk} FAILED: {e}")
            gc.collect()
            if device == 'cuda':
                torch.cuda.empty_cache()

    # Save raw results
    with open(os.path.join(args.output, 'pythia_results.json'), 'w') as f:
        json.dump(results, f, indent=2)

    # Head-to-head summary table (s0 / beta / R2 per method)
    print(f"\n{'═'*104}")
    hdr = f"  {'Model':<8} {'Params':>8} {'L':>3} {'d_ff':>6} {'PPL':>8}"
    for m in METHODS:
        hdr += f" | {m+' s0':>9} {m+' β':>7} {m+' R²':>6}"
    print(hdr)
    print(f"{'─'*104}")
    for r in sorted(results, key=lambda x: x['n_params_M']):
        line = (f"  {r['model']:<8} {r['n_params_M']:>7.0f}M {r['L']:>3} "
                f"{r['d_ff']:>6} {r['ppl_baseline']:>8.1f}")
        for m in METHODS:
            md = r['methods'].get(m, {})
            if md.get('sigmoid_R2') is not None:
                line += (f" | {md['sigmoid_s_0']:>9.3f} {md['sigmoid_beta']:>7.2f} "
                         f"{md['sigmoid_R2']:>6.3f}")
            else:
                line += f" | {'FAILED':>9} {'':>7} {'':>6}"
        print(line)
    print(f"{'═'*104}")

    # Per-method scaling laws
    scalings = {}
    for m in METHODS:
        scalings[m] = fit_scaling_laws(results, m)
        if scalings[m]:
            with open(os.path.join(args.output,
                                   f'pythia_scaling_laws_{m}.json'), 'w') as f:
                json.dump(scalings[m], f, indent=2)

    # Plots
    print(f"\n  Generating plots ...")
    make_plots(results, scalings, args.output)

    dt = time.time() - t_total
    print(f"\n  Total runtime: {dt/60:.1f} min")
    print("  Done!")


if __name__ == '__main__':
    main()


## Retraining after pruning — is BASP's sparsity advantage real?

The one-shot sweep above shows BASP reaching a **lower critical density** `s_0`
than WANDA (it can keep fewer neurons before perplexity collapses). But a lower
one-shot `s_0` could be an artifact of *which* neurons survive being a better
starting point for the dense forward pass — not of the pruned network being
fundamentally more trainable. Fine-tuning is the standard test: if BASP's
advantage survives a short recovery train, it is structural; if WANDA catches
up once both are retrained, the advantage was only about the one-shot snapshot.

**Experiment (Pythia-1.4b — where BASP's margin is largest).** For each method
we prune the MLP intermediate neurons *at that method's own critical density*
(forward-hook structured pruning, identical mechanism as above), then run a
short recovery fine-tune of the surviving network and re-measure WikiText-2
perplexity **and** next-token accuracy:

| method | density kept `s_0` | neurons removed |
|--------|-------------------|-----------------|
| WANDA  | 0.976             | 2.4%            |
| BASP   | 0.660             | 34%             |

So BASP enters retraining having removed **~14× more neurons** than WANDA. The
hypothesis test:

* **BASP advantage is real** → after retraining BASP matches (or nearly matches)
  WANDA's perplexity/accuracy *despite* having thrown away far more capacity.
* **BASP advantage is an artifact** → retrained BASP lags WANDA, i.e. once you
  are allowed to fine-tune, the gentler WANDA prune recovers to a strictly
  better model.

During fine-tuning the same post-GELU mask stays applied, so pruned neurons
receive zero gradient (their up-projection rows and down-projection columns are
frozen at their pre-prune values and never contribute) — we train only the
surviving structure. Restore-from-pristine between methods keeps the two runs
independent.

> **Colab memory note.** Full fine-tuning of 1.4b needs an A100/L4-class GPU.
> On a 16 GB T4, set `RETRAIN_CONFIG['train_scope'] = 'mlp'` (trains only the
> MLP weights being pruned) and/or lower `seq_len`/`steps`. An 8-bit optimizer
> (`bitsandbytes`) is used automatically when available.


In [ ]:
# ===========================================================
#  RETRAINING-AFTER-PRUNING EXPERIMENT  (config + helpers)
# ===========================================================

RETRAIN_CONFIG = dict(
    model='1.4b',                              # BASP's margin is largest here
    densities={'wanda': 0.976, 'basp': 0.660}, # keep-fraction s_0 per method
    steps=300,            # optimizer-seen sequences (recovery fine-tune)
    lr=5e-5,
    seq_len=512,
    grad_accum=4,
    n_train_texts=1024,   # text pool drawn for fine-tuning
    train_scope='all',    # 'all' = full model | 'mlp' = only MLP weights (T4-safe)
    grad_checkpoint=True,
    weight_decay=0.0,     # 0 so frozen/masked neurons never drift via decay
)


def load_text_samples(n):
    """Draw ``n`` raw text samples (C4 stream, wikitext-2-train fallback)."""
    from datasets import load_dataset
    texts = []
    try:
        c4 = load_dataset('allenai/c4', 'en', split='train', streaming=True)
        for s in c4:
            if s['text'].strip():
                texts.append(s['text'])
            if len(texts) >= n:
                break
    except Exception as e:
        print(f"  (C4 unavailable: {e}; using wikitext-2 train)")
        wt = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='train')
        for t in wt['text']:
            if t.strip():
                texts.append(t)
            if len(texts) >= n:
                break
    return texts


def evaluate_ppl_acc(model, tokenizer, dataset=EVAL_DATASET,
                     max_tokens=EVAL_MAX_TOKENS, stride=EVAL_STRIDE,
                     max_length=EVAL_MAX_LENGTH, device='cuda'):
    """Sliding-window WikiText-2 perplexity AND next-token top-1 accuracy in a
    single pass (so retraining eval costs one forward sweep, not two)."""
    from datasets import load_dataset
    if dataset == 'wikitext':
        test = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='test')
        text = '\n\n'.join(test['text'])
    else:
        ds = load_dataset('NeelNanda/pile-10k', split='train')
        text = '\n\n'.join(ds['text'][:300])

    enc = tokenizer(text, return_tensors='pt')
    seq_len = min(enc.input_ids.size(1), max_tokens)

    nll_sum, n_tokens, correct, acc_total, prev_end = 0.0, 0, 0, 0, 0
    for begin in range(0, seq_len, stride):
        end = min(begin + max_length, seq_len)
        trg_len = end - prev_end
        input_ids = enc.input_ids[:, begin:end].to(device)
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100

        with torch.no_grad():
            out = model(input_ids, labels=target_ids)
        loss, logits = out.loss, out.logits

        num_scored = (target_ids != -100).sum().item() - 1
        if num_scored > 0:
            nll_sum += loss.float().item() * num_scored
            n_tokens += num_scored

        shift_logits = logits[:, :-1, :]
        shift_tgt = target_ids[:, 1:]
        m = shift_tgt != -100
        if m.any():
            preds = shift_logits.argmax(dim=-1)
            correct += (preds[m] == shift_tgt[m]).sum().item()
            acc_total += int(m.sum().item())

        prev_end = end
        if end >= seq_len:
            break

    ppl = float(np.exp(nll_sum / n_tokens)) if n_tokens else float('inf')
    acc = (correct / acc_total) if acc_total else 0.0
    return ppl, acc


def finetune_pruned(model, tokenizer, pruner, cfg, device='cuda'):
    """Short recovery fine-tune of a structurally-pruned model.

    ``pruner`` stays enabled throughout: the post-GELU mask zeroes both the
    forward contribution and the gradient of pruned neurons, so only the
    surviving structure is trained. Returns the optimizer name used.
    """
    pruner.enable()
    model.train()
    model.config.use_cache = False
    if cfg['grad_checkpoint']:
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={'use_reentrant': False})

    if cfg['train_scope'] == 'mlp':
        for n, p in model.named_parameters():
            p.requires_grad = ('.mlp.' in n)
    else:
        for p in model.parameters():
            p.requires_grad = True
    trainable = [p for p in model.parameters() if p.requires_grad]
    n_train = sum(p.numel() for p in trainable) / 1e6

    try:
        import bitsandbytes as bnb
        opt = bnb.optim.AdamW8bit(trainable, lr=cfg['lr'],
                                  weight_decay=cfg['weight_decay'])
        opt_name = 'AdamW8bit'
    except Exception:
        opt = torch.optim.AdamW(trainable, lr=cfg['lr'],
                                weight_decay=cfg['weight_decay'])
        opt_name = 'AdamW'
    print(f"    optimizer={opt_name}  trainable={n_train:.0f}M params "
          f"(scope={cfg['train_scope']})")

    texts = load_text_samples(cfg['n_train_texts'])
    accum = cfg['grad_accum']
    step, running = 0, 0.0
    opt.zero_grad()
    t0 = time.time()
    while step < cfg['steps']:
        for text in texts:
            ids = tokenizer(text, return_tensors='pt', truncation=True,
                            max_length=cfg['seq_len']).to(device)
            if ids.input_ids.size(1) < 2:
                continue
            loss = model(**ids, labels=ids.input_ids).loss / accum
            loss.backward()
            running += loss.item() * accum
            if (step + 1) % accum == 0:
                torch.nn.utils.clip_grad_norm_(trainable, 1.0)
                opt.step()
                opt.zero_grad()
            step += 1
            if step % 50 == 0:
                print(f"    step {step:>4}/{cfg['steps']}  "
                      f"loss={running/50:.4f}  [{time.time()-t0:.0f}s]")
                running = 0.0
            if step >= cfg['steps']:
                break

    model.eval()
    model.config.use_cache = True
    if cfg['grad_checkpoint']:
        model.gradient_checkpointing_disable()
    del opt
    gc.collect()
    if device == 'cuda':
        torch.cuda.empty_cache()
    return opt_name


In [ ]:
def run_retrain_experiment(token, cfg=RETRAIN_CONFIG, device='cuda'):
    """Prune Pythia-1.4b at each method's critical density, fine-tune the
    surviving network, and compare one-shot vs retrained perplexity/accuracy.

    Returns a dict of results and prints a head-to-head table.
    """
    model_key = cfg['model']
    spec = PYTHIA_MODELS[model_key]
    hf_name, d_ff = spec['hf'], spec['d_ff']

    print('=' * 72)
    print(f"  RETRAIN-AFTER-PRUNE  —  {hf_name}  (d_ff={d_ff})")
    print('=' * 72)

    model = AutoModelForCausalLM.from_pretrained(
        hf_name, torch_dtype=torch.float16, token=token,
        low_cpu_mem_usage=True).to(device)
    model.eval()
    tokenizer = AutoTokenizer.from_pretrained(hf_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # --- dense reference ----------------------------------------------------
    print("  Dense baseline ...", end='', flush=True)
    ppl_dense, acc_dense = evaluate_ppl_acc(model, tokenizer, device=device)
    print(f"  ppl={ppl_dense:.2f}  acc={acc_dense:.4f}")

    # --- calibrate once + score both methods --------------------------------
    print(f"  Calibrating ({N_CALIB_SAMPLES} samples) ...", end='', flush=True)
    cal_texts = load_text_samples(N_CALIB_SAMPLES)
    collector = MLPActivationCollector(model)
    collector.register()
    with torch.no_grad():
        for text in cal_texts:
            ids = tokenizer(text, return_tensors='pt', truncation=True,
                            max_length=CALIB_SEQ_LEN).to(device)
            model(**ids)
    collector.remove()
    input_norms, inter_norms = collector.get_norms()
    del collector
    method_scores = {m: SCORERS[m](model, input_norms, inter_norms)
                     for m in cfg['densities']}
    del input_norms, inter_norms
    print(" done")

    # Pristine weights (CPU) so each method retrains independently.
    pristine = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    results = {'model': model_key, 'hf_name': hf_name, 'd_ff': d_ff,
               'ppl_dense': ppl_dense, 'acc_dense': acc_dense, 'methods': {}}

    for method, density in cfg['densities'].items():
        print(f"\n  {'─'*60}\n  [{method.upper()}]  keep s_0={density:.3f}  "
              f"(remove {1-density:.1%}, {d_ff-int(round(d_ff*density))}/{d_ff} neurons)")
        model.load_state_dict(pristine)          # restore for an independent run
        model.eval()
        scores = method_scores[method]

        # one-shot (no fine-tune)
        pruner = TopKPruner(model, scores, density=density)
        pruner.enable()
        ppl_os, acc_os = evaluate_ppl_acc(model, tokenizer, device=device)
        pruner.disable()
        print(f"    one-shot : ppl={ppl_os:.2f}  acc={acc_os:.4f}")

        # recovery fine-tune (fresh pruner with identical masks)
        pruner = TopKPruner(model, scores, density=density)
        opt_name = finetune_pruned(model, tokenizer, pruner, cfg, device=device)
        pruner.enable()                          # keep mask applied for eval
        ppl_ft, acc_ft = evaluate_ppl_acc(model, tokenizer, device=device)
        pruner.disable()
        print(f"    retrained: ppl={ppl_ft:.2f}  acc={acc_ft:.4f}  "
              f"(Δppl={ppl_ft-ppl_os:+.2f}, Δacc={acc_ft-acc_os:+.4f})")

        results['methods'][method] = dict(
            density=density, neurons_removed=d_ff - int(round(d_ff * density)),
            ppl_oneshot=ppl_os, acc_oneshot=acc_os,
            ppl_retrained=ppl_ft, acc_retrained=acc_ft, optimizer=opt_name)

    # --- head-to-head -------------------------------------------------------
    print(f"\n{'═'*72}")
    print(f"  Dense 1.4b:  ppl={ppl_dense:.2f}  acc={acc_dense:.4f}")
    print(f"  {'method':<7} {'keep':>6} {'rm':>5} | {'ppl 1-shot':>11} {'ppl retrn':>10}"
          f" | {'acc 1-shot':>11} {'acc retrn':>10}")
    print(f"  {'─'*70}")
    for m, md in results['methods'].items():
        print(f"  {m:<7} {md['density']:>6.3f} {1-md['density']:>5.0%} | "
              f"{md['ppl_oneshot']:>11.2f} {md['ppl_retrained']:>10.2f} | "
              f"{md['acc_oneshot']:>11.4f} {md['acc_retrained']:>10.4f}")
    print(f"{'═'*72}")
    if {'wanda', 'basp'} <= set(results['methods']):
        w, b = results['methods']['wanda'], results['methods']['basp']
        dppl = b['ppl_retrained'] - w['ppl_retrained']
        dacc = b['acc_retrained'] - w['acc_retrained']
        print(f"  After retraining, BASP (removed {b['neurons_removed']}) vs "
              f"WANDA (removed {w['neurons_removed']}):")
        print(f"    Δppl(basp-wanda)={dppl:+.2f}   Δacc(basp-wanda)={dacc:+.4f}")
        if dppl <= 0.5 and dacc >= -0.005:
            print("    -> BASP matches/beats WANDA despite removing far more "
                  "neurons: the sparsity advantage SURVIVES retraining.")
        else:
            print("    -> retrained WANDA is better: BASP's one-shot edge is "
                  "partly an artifact of the one-shot protocol.")

    with open(os.path.join(OUTPUT_DIR, 'pythia_retrain_1.4b.json'), 'w') as f:
        json.dump(results, f, indent=2)
    print(f"\n  Saved -> {os.path.join(OUTPUT_DIR, 'pythia_retrain_1.4b.json')}")

    del model, tokenizer, method_scores, pristine
    gc.collect()
    if device == 'cuda':
        torch.cuda.empty_cache()
    return results


# Run the retraining experiment (prompts once for the HF token, like main()).
os.makedirs(OUTPUT_DIR, exist_ok=True)
_device = 'cuda' if torch.cuda.is_available() else 'cpu'
_token = os.environ.get('HF_TOKEN') or getpass('Enter HuggingFace token: ')
retrain_results = run_retrain_experiment(_token, device=_device)
